## Image Descriptions with Gemini 

Generate detailed textual descriptions for extracted images using Gemini 2.5 Flash.

**Prerequisites:**
- Make sure you rag-data dir with extracted dir like markdown, images and tables
- Google API key set in .env file

**Output:**
- Markdown descriptions saved to `data/rag-data/images_desc/{company}/{document}/page_X.md`

### Setup and Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from PIL import Image
import base64
import io

### Configuration

In [ ]:
# Paths
IMAGES_DIR = "data/rag-data/images"
OUTPUT_DESC_DIR = "data/rag-data/images_desc"

# Model configuration
MODEL_NAME = "gemini-2.5-flash-lite"
# MODEL_NAME = "ministral-3"

model = ChatGoogleGenerativeAI(model=MODEL_NAME)
# model = ChatOllama(model=MODEL_NAME, temperature=0)

### Description Generation Function

In [3]:
describe_image_prompt = """Analyze this financial document page and extract meaningful data in a concise format.

For charts and graphs:
- Identify the metric being measured
- List key data points and values
- Note significant trends (growth, decline, stability)

For tables:
- Extract column headers and key rows
- Note important values and totals

For text:
- Summarize key facts and numbers only
- Skip formatting, headers, and navigation elements

Be direct and factual. Focus on numbers, trends, and insights that would be useful for retrieval."""

In [4]:
from langchain.messages import SystemMessage


def generate_image_description(image_path: Path):
    image = Image.open(image_path)
    buffered = io.BytesIO()
    image.save(buffered, format='PNG')

    image_base64 = base64.b64encode(buffered.getvalue()).decode()

    message = HumanMessage(
        content=[
            {'type': 'text', 'text': describe_image_prompt},
            {
                'type': 'image_url',
                'image_url': {'url': f"data:image/png;base64,{image_base64}"}
            }
        ]
    )
    system_prompt = SystemMessage('You are an AI Assistant')

    response = model.invoke([system_prompt, message])

    return response.content

In [5]:
image_path = Path(r'data\rag-data\images\meta\meta 10-k 2024\page_64.png')

response = generate_image_description(image_path)

In [6]:
print(response)

def generate_and_save_description(image_path: Path):
    company_name = image_path.parent.parent.name
    doc_name = image_path.parent.name

    output_dir = Path(OUTPUT_DESC_DIR)/company_name/doc_name
    output_dir.mkdir(parents=True, exist_ok=True)

    desc_file = output_dir / f"{image_path.stem}.md"

    if desc_file.exists():
        return False
    
    description = generate_image_description(image_path)
    desc_file.write_text(description, encoding='utf-8')
    
    return True

### **Analysis of Revenue by User Geography**

#### **Key Insights from Text**
*   **Revenue Reporting:** Revenue is reported based on the geography where impressions are delivered, sold, or purchased.
*   **2023 Regional Breakdown:** 
    *   United States & Canada: 28%
    *   Asia-Pacific: 31% (implied by text context)
    *   Rest of World: [Value not explicitly totaled in single figure but summarized as "rest"]
*   **2024 Growth Rates:**
    *   United States & Canada: +18%
    *   Next segment (Asia-Pacific): +20% 
    *   Rest of World: +11%

#### **Revenue Metrics by Geography (Oct 2022 – Dec 2024)**
The data is divided into **Ad Revenue** (blue) and **Non-Ad Revenue** (gray). All charts show a consistent upward trend in both categories.

| Region | Ad Revenue Trend | Non-Ad Revenue Trend | Key Observations |
| :--- | :--- | :--- | :--- |
| **United States & Canada** | Significant Growth | Steady Growth | Largest volume; steady growth in non-ad revenue contrasted with high-grow

In [7]:
image_path = Path(r'data\rag-data\images\meta\meta 10-k 2024\page_64.png')

response = generate_and_save_description(image_path)

In [8]:
from tqdm import tqdm

images_path = Path(IMAGES_DIR)
image_files = list(images_path.rglob("page_*.png"))

for image_path in tqdm(image_files):
    response = generate_and_save_description(image_path)


100%|██████████| 77/77 [00:00<00:00, 3289.62it/s]
